# BR benchmark (BR1–BR15)

## Utilities

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from settings import EXPERIMENTS_FOLDER

SETS = [f"BR{k}" for k in range(1, 16)]
GROUPS = {"Avg. (BR1–BR7)": SETS[:7], "Avg. (BR8–BR15)": SETS[7:], "Avg. (BR1–BR15)": SETS}
SEEDS = [42, 43, 44]

def instance_vols(name):
    """Volume per (set, instance) from <name>_BR{k}.csv. If the name contains '{seed}', seeds are averaged.
    Rounded to 4 decimals because BSG-VCS results are stored with 6 decimals."""
    if "{seed}" in name:
        return pd.concat([instance_vols(name.format(seed=seed)) for seed in SEEDS], axis=1).mean(axis=1)
    vols = {}
    for s in SETS:
        df = pd.read_csv(EXPERIMENTS_FOLDER / f"{name}_{s}.csv")
        vols[s] = df.set_index(df.columns[0] if df.columns[0] == "instance" else "Instancia")[
            next(c for c in df.columns if c.startswith("Vol"))]
    return pd.concat(vols, names=["set", "instance"]).round(4)

def read(name, metric="Vol"):
    """Mean per BR set from <name>_BR{k}.csv. If the name contains '{seed}', seeds are averaged."""
    if "{seed}" in name:
        return pd.concat([read(name.format(seed=seed), metric) for seed in SEEDS], axis=1).mean(axis=1)
    means = {}
    for s in SETS:
        df = pd.read_csv(EXPERIMENTS_FOLDER / f"{name}_{s}.csv")
        means[s] = df[next(c for c in df.columns if c.startswith(metric))].mean()
    return pd.Series(means)

def with_groups(table):
    """Appends the average rows of each group of sets (all sets have the same number of instances)."""
    return pd.concat([table, pd.DataFrame({g: table.loc[sets].mean() for g, sets in GROUPS.items()}).T])

def loss_gap(vol, ref):
    """Loss Gap (%) computed from the average volume of each row."""
    return ((100 - ref) - (100 - vol)) / (100 - ref) * 100

def scoreboard(label, a, b):
    """Per-instance count of a against b and Wilcoxon signed-rank test."""
    d = a - b
    print(f"{label} | better: {(d > 0).sum()}, equal: {(d == 0).sum()}, worse: {(d < 0).sum()} "
          f"| Wilcoxon: p = {wilcoxon(d[d != 0]).pvalue:.2e}")

def per_seed_gap(prefix, ref):
    """Loss Gap of each seed of <prefix>_S{seed} in the average rows."""
    return pd.DataFrame({
        f"S{seed}": loss_gap(with_groups(read(f"{prefix}_S{seed}").to_frame())[0], ref) for seed in SEEDS
    }).loc[list(GROUPS)]

def bold_max(row):
    return ["font-weight: bold" if v == row.max() else "" for v in row]

## Model comparison

In [2]:
# Loss Gap (%) of each model (averaged over 3 seeds): Greedy w.r.t. VCS and BSG-Hybrid (w=8) w.r.t. BSG-VCS (w=8)
models = {"M3": "M3", "M30": "M30", "M100": "M100", "M-Mix": "MIX_ENC4"}

vcs = with_groups(read("vcs/vcs").to_frame())[0]
bsg_vcs = with_groups(read("bsg_vcs/bsg_vcs_w8").to_frame())[0]
greedy = with_groups(pd.DataFrame({m: read(f"fast/greedy_fast_{p}_S{{seed}}") for m, p in models.items()}))
hybrid = with_groups(pd.DataFrame({m: read(f"fast/bsg_hybrid_fast_{p}_S{{seed}}") for m, p in models.items()}))

table = pd.concat({
    "Greedy": greedy.apply(loss_gap, ref=vcs),
    "BSG-Hybrid (w=8)": hybrid.apply(loss_gap, ref=bsg_vcs),
}, axis=1)

strategies = {"Greedy": ("fast/greedy_fast", vcs), "BSG-Hybrid (w=8)": ("fast/bsg_hybrid_fast", bsg_vcs)}
per_seed = {(strategy, m): per_seed_gap(f"{prefix}_{p}", ref)
            for strategy, (prefix, ref) in strategies.items() for m, p in models.items()}
print("Standard deviation of the Loss Gap across seeds:")
print(pd.DataFrame({k: v.std(axis=1) for k, v in per_seed.items()}).round(2).to_string())

# M-Mix vs M30 on BR1–BR15 under BSG-Hybrid: number of seed pairs won by M-Mix
mix = per_seed[("BSG-Hybrid (w=8)", "M-Mix")].loc["Avg. (BR1–BR15)"]
m30 = per_seed[("BSG-Hybrid (w=8)", "M30")].loc["Avg. (BR1–BR15)"]
print(f"\nBSG-Hybrid, BR1–BR15 | M-Mix: {mix.round(2).to_dict()} | M30: {m30.round(2).to_dict()}")
print(f"M-Mix outperforms M30 in {sum(a > b for a in mix for b in m30)}/{len(mix) * len(m30)} seed pairs\n")

(table.style.format("{:.2f}")
    .apply(bold_max, axis=1, subset=["Greedy"])
    .apply(bold_max, axis=1, subset=["BSG-Hybrid (w=8)"]))

Standard deviation of the Loss Gap across seeds:
                Greedy                   BSG-Hybrid (w=8)                  
                    M3   M30  M100 M-Mix               M3   M30  M100 M-Mix
Avg. (BR1–BR7)    1.13  0.59  0.34  0.79             0.32  0.59  0.31  0.21
Avg. (BR8–BR15)   3.15  0.61  0.14  0.14             0.30  0.22  0.08  0.25
Avg. (BR1–BR15)   2.15  0.47  0.24  0.46             0.31  0.38  0.15  0.12

BSG-Hybrid, BR1–BR15 | M-Mix: {'S42': 4.45, 'S43': 4.65, 'S44': 4.45} | M30: {'S42': 4.2, 'S43': 3.88, 'S44': 4.63}
M-Mix outperforms M30 in 7/9 seed pairs



## Performance evaluation

In [3]:
# Average volume utilization (%), time per instance (s) and Loss Gap (%) of the median model (sequential evaluations, run_eval)
methods = {
    "VCS": "vcs/vcs",
    "Greedy": "greedy/greedy_MIX_MEDIAN",
    "BSG-VCS (w=8)": "bsg_vcs/bsg_vcs_w8",
    "BSG-VCS (w=16)": "bsg_vcs/bsg_vcs_w16",
    "BSG-Hybrid (w=8)": "bsg_hybrid/bsg_hybrid_MIX_MEDIAN_w8",
    "BSG-Hybrid (w=16)": "bsg_hybrid/bsg_hybrid_MIX_MEDIAN_w16",
}
# Loss Gap reference of each method: Greedy over VCS and BSG-Hybrid over BSG-VCS at equal w
references = {"Greedy": "VCS", "BSG-Hybrid (w=8)": "BSG-VCS (w=8)", "BSG-Hybrid (w=16)": "BSG-VCS (w=16)"}

vol = with_groups(pd.DataFrame({m: read(name) for m, name in methods.items()}))
time = with_groups(pd.DataFrame({m: read(name, "Time") for m, name in methods.items()}))
gap = pd.DataFrame({m: loss_gap(vol[m], vol[ref]) for m, ref in references.items()}).loc[list(GROUPS)]

time = time.loc[["Avg. (BR1–BR7)", "Avg. (BR8–BR15)"]].rename(index=lambda g: g.replace("Avg. (", "Time (s, "))
gap = gap.rename(index=lambda g: g.replace("Avg.", "ΔL"))

# Per-instance scoreboard of BSG-Hybrid against BSG-VCS at equal w (n = 1500)
for w in [8, 16]:
    scoreboard(f"BSG-Hybrid vs BSG-VCS (w={w})",
               instance_vols(methods[f"BSG-Hybrid (w={w})"]), instance_vols(methods[f"BSG-VCS (w={w})"]))

table = pd.concat({"Vol (%)": vol, "Time (s)": time, "ΔL (%)": gap}).reindex(columns=list(methods))
(table.style.format("{:.3f}", subset=pd.IndexSlice["Time (s)", :], na_rep="")
    .format("{:.2f}", subset=pd.IndexSlice[["Vol (%)", "ΔL (%)"], :], na_rep=""))

BSG-Hybrid vs BSG-VCS (w=8) | better: 1017, equal: 9, worse: 474 | Wilcoxon: p = 3.42e-55
BSG-Hybrid vs BSG-VCS (w=16) | better: 942, equal: 9, worse: 549 | Wilcoxon: p = 1.05e-36


## Scalability

In [4]:
# Average volume utilization (%) and time per instance (s) per BR set, for each beam width w (sequential evaluations, run_eval)
widths = [2, 4, 8, 16, 32]
strategies = {"BSG-VCS": "bsg_vcs/bsg_vcs_w{w}", "BSG-Hybrid": "bsg_hybrid/bsg_hybrid_MIX_MEDIAN_w{w}"}

tables = {
    strategy: with_groups(pd.concat({
        f"w={w}": pd.DataFrame({"Vol (%)": read(name.format(w=w)), "Time (s)": read(name.format(w=w), "Time")})
        for w in widths
    }, axis=1))
    for strategy, name in strategies.items()
}

for strategy, table in tables.items():
    fmt = {c: "{:.2f}" if c[1] == "Vol (%)" else "{:.3f}" for c in table.columns}
    display(table.style.set_caption(strategy).format(fmt))

# Loss Gap (%) of BSG-Hybrid w.r.t. BSG-VCS at equal w, on BR1–BR15
avg = "Avg. (BR1–BR15)"
gaps = pd.DataFrame({
    f"w={w}": [loss_gap(tables["BSG-Hybrid"].loc[avg, (f"w={w}", "Vol (%)")], tables["BSG-VCS"].loc[avg, (f"w={w}", "Vol (%)")])]
    for w in widths
}, index=["ΔL (%, BR1–BR15)"])
display(gaps.style.set_caption("BSG-Hybrid vs BSG-VCS").format("{:.2f}"))

,w=2,w=4,w=8,w=16,w=32
"ΔL (%, BR1–BR15)",1.10,4.32,4.45,3.26,3.27


## Time limits

In [5]:
# Average volume utilization (%) per BR set for each time limit and method (iterative scheme of BSG-VCS)
times = [30, 150]
strategies = {"BSG-VCS": "de_bsg_vcs/de_bsg_vcs_t{t}", "BSG-Hybrid": "de_bsg_hybrid/de_bsg_hybrid_MIX_MEDIAN_t{t}"}

de_vol = with_groups(pd.DataFrame({
    (f"{t}s", strategy): read(name.format(t=t)) for t in times for strategy, name in strategies.items()
}))

# Per-instance scoreboard of BSG-Hybrid against BSG-VCS under the same time limit (n = 1500)
for t in times:
    scoreboard(f"BSG-Hybrid vs BSG-VCS ({t}s)",
               instance_vols(strategies["BSG-Hybrid"].format(t=t)), instance_vols(strategies["BSG-VCS"].format(t=t)))

display(de_vol.style.format("{:.2f}").apply(bold_max, axis=1, subset=["30s"]).apply(bold_max, axis=1, subset=["150s"]))

# Loss Gap (%) of BSG-Hybrid w.r.t. BSG-VCS under the same time limit, per group of sets
de_gap = pd.DataFrame({
    f"{t}s": loss_gap(de_vol.loc[list(GROUPS), (f"{t}s", "BSG-Hybrid")], de_vol.loc[list(GROUPS), (f"{t}s", "BSG-VCS")])
    for t in times
}).rename(index=lambda g: g.replace("Avg.", "ΔL"))
display(de_gap.style.set_caption("BSG-Hybrid vs BSG-VCS").format("{:.2f}"))

BSG-Hybrid vs BSG-VCS (30s) | better: 1035, equal: 26, worse: 439 | Wilcoxon: p = 7.59e-76
BSG-Hybrid vs BSG-VCS (150s) | better: 985, equal: 30, worse: 485 | Wilcoxon: p = 1.21e-49


,30s,150s
ΔL (BR1–BR7),3.08,2.22
ΔL (BR8–BR15),5.35,3.71
ΔL (BR1–BR15),4.44,3.11


## Loss weighting analysis

In [6]:
# Loss Gap (%) w.r.t. BSG-VCS (w=8) of M-Mix under four loss weighting schemes (averaged over 3 seeds), BSG-Hybrid (w=8)
configs = {
    "Hard-biased (.25/.25/.50)": "MIX_ENC4",
    "Medium-biased (.25/.50/.25)": "MIX_WMEDIUM",
    "Easy-biased (.50/.25/.25)": "MIX_WEASY",
    "Balanced (.33/.33/.33)": "MIX_WBAL",
}

bsg_vcs = with_groups(read("bsg_vcs/bsg_vcs_w8").to_frame())[0]
hybrid = with_groups(pd.DataFrame({c: read(f"fast/bsg_hybrid_fast_{p}_S{{seed}}") for c, p in configs.items()}))
table = hybrid.apply(loss_gap, ref=bsg_vcs)

per_seed = {c: per_seed_gap(f"fast/bsg_hybrid_fast_{p}", bsg_vcs) for c, p in configs.items()}
print("Standard deviation of the Loss Gap across seeds:")
print(pd.DataFrame({c: v.std(axis=1) for c, v in per_seed.items()}).round(2).to_string())

# Hard-biased vs Balanced on BR1–BR15: number of seed pairs won by Hard-biased
hard = per_seed["Hard-biased (.25/.25/.50)"].loc["Avg. (BR1–BR15)"]
bal = per_seed["Balanced (.33/.33/.33)"].loc["Avg. (BR1–BR15)"]
print(f"\nBR1–BR15 | Hard-biased: {hard.round(2).to_dict()} | Balanced: {bal.round(2).to_dict()}")
print(f"Hard-biased outperforms Balanced in {sum(a > b for a in hard for b in bal)}/{len(hard) * len(bal)} seed pairs\n")

table.style.format("{:.2f}").apply(bold_max, axis=1)

Standard deviation of the Loss Gap across seeds:
                 Hard-biased (.25/.25/.50)  Medium-biased (.25/.50/.25)  Easy-biased (.50/.25/.25)  Balanced (.33/.33/.33)
Avg. (BR1–BR7)                        0.21                         0.28                       0.29                    0.08
Avg. (BR8–BR15)                       0.25                         0.37                       0.02                    0.16
Avg. (BR1–BR15)                       0.12                         0.09                       0.14                    0.11

BR1–BR15 | Hard-biased: {'S42': 4.45, 'S43': 4.65, 'S44': 4.45} | Balanced: {'S42': 4.07, 'S43': 3.88, 'S44': 3.88}
Hard-biased outperforms Balanced in 9/9 seed pairs



,Hard-biased (.25/.25/.50),Medium-biased (.25/.50/.25),Easy-biased (.50/.25/.25),Balanced (.33/.33/.33)
BR1,5.36,3.84,5.16,5.57
BR2,2.95,3.47,3.27,3.64
BR3,4.51,4.10,3.91,4.39
BR4,3.67,2.92,2.94,3.43
BR5,2.42,2.16,1.46,1.68
BR6,2.51,1.59,1.98,3.11
BR7,3.83,2.28,2.84,2.97
BR8,7.01,5.10,5.02,6.35
BR9,6.83,4.59,4.84,5.31
BR10,4.68,3.22,3.88,3.65


## Ablation: container stream

In [7]:
# With and without the container stream (M-Mix vs M-Mix without container stream), BSG-Hybrid (w=8), 3 seeds
configs = {"With container": "MIX_ENC4", "Without container": "MIX_NOCONT"}

# Paired per-instance test (n = 1500), with the volume averaged over the seeds
vols = pd.DataFrame({name: instance_vols(f"fast/bsg_hybrid_fast_{p}_S{{seed}}") for name, p in configs.items()})
scoreboard("With vs without container", vols.iloc[:, 0], vols.iloc[:, 1])

# Loss Gap of each seed w.r.t. BSG-VCS (w=8)
bsg_vcs = with_groups(read("bsg_vcs/bsg_vcs_w8").to_frame())[0]
per_seed = {name: per_seed_gap(f"fast/bsg_hybrid_fast_{p}", bsg_vcs) for name, p in configs.items()}

cont = per_seed["With container"].loc["Avg. (BR1–BR15)"]
nocont = per_seed["Without container"].loc["Avg. (BR1–BR15)"]
print(f"\nBR1–BR15 | With container: {cont.round(2).to_dict()} | Without container: {nocont.round(2).to_dict()}")
print(f"With container outperforms without container in {sum(a > b for a in cont for b in nocont)}/{len(cont) * len(nocont)} seed pairs\n")

# Loss Gap across seeds (mean ± standard deviation)
pd.DataFrame({name: v.mean(axis=1).map("{:.2f}".format) + " ± " + v.std(axis=1).map("{:.2f}".format)
              for name, v in per_seed.items()})

With vs without container | better: 775, equal: 16, worse: 709 | Wilcoxon: p = 3.56e-03

BR1–BR15 | With container: {'S42': 4.45, 'S43': 4.65, 'S44': 4.45} | Without container: {'S42': 4.19, 'S43': 4.33, 'S44': 3.9}
With container outperforms without container in 9/9 seed pairs



,With container,Without container
Avg. (BR1–BR7),3.68 ± 0.21,3.75 ± 0.52
Avg. (BR8–BR15),5.17 ± 0.25,4.44 ± 0.07
Avg. (BR1–BR15),4.51 ± 0.12,4.14 ± 0.22


## Ablation: scoring function and expansion rule

In [8]:
# Average volume utilization (%) per BR set: scoring function (VCS / Linear / Proposed model) x expansion rule (fixed / dynamic), w=8
from models.vcs_linear import VCSLinear
from training.training import load_model

configs = {
    ("VCS", "Fixed"): "fast/bsg_vcs_fixedw_fast",
    ("VCS", "Dynamic"): "fast/bsg_vcs_fast",
    ("Linear", "Fixed"): "fast/bsg_hybrid_fast_LINEAR_MEDIAN",
    ("Linear", "Dynamic"): "fast/bsg_hybrid_dynw_fast_LINEAR_MEDIAN",
    ("Proposed model", "Fixed"): "fast/bsg_hybrid_fast_MIX_MEDIAN",
    ("Proposed model", "Dynamic"): "fast/bsg_hybrid_dynw_fast_MIX_MEDIAN",
}

# Exponents learned by the linear model (normalized by the weight of log V), compared with those of VCS
exponents = pd.DataFrame({f"Linear S{seed}": load_model(VCSLinear, f"LINEAR_S{seed}").exponents() for seed in SEEDS})
exponents["VCS"] = {"delta": 1.0, "alpha": 4.0, "beta": 1.0, "gamma": 0.2}
print(exponents.T.round(3).to_string(), "\n")

grid = with_groups(pd.DataFrame({key: read(prefix) for key, prefix in configs.items()}))
grid.columns.names = ["Scoring function", "Expansion rule"]
grid.style.format("{:.2f}")

            delta  alpha   beta  gamma
Linear S42    1.0  2.895  1.929  0.541
Linear S43    1.0  2.934  1.999  0.545
Linear S44    1.0  2.869  2.030  0.551
VCS           1.0  4.000  1.000  0.200 

